In [ ]:
import pandas as pd
import numpy as np
import regex as rex

import emoji
import re

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, models

In [ ]:
#model = SentenceTransformer('all-mpnet-base-v2')  # schnell + leicht
word_embedding_model = models.Transformer("vinai/bertweet-base")
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

RobertaModel LOAD REPORT from: vinai/bertweet-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.weight            | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
chunks = pd.read_csv(
    "bitcoin_tweets_clean.csv",
    chunksize=100_000,
    engine="python",          # 🔴 extrem wichtig
)

In [ ]:
df = pd.concat(chunks)

In [ ]:
for y, text in enumerate(df["text"]):
  if emoji.emoji_count(text) > 0:
    print(y, df["text"].iloc[y])
    break

13 :fireworks: Happy New Year :fireworks: 2020 Asset Performance: + 300% : #Bitcoin + 44%. : Nasdaq :classical_building_selector: + 25%. : Gold 🪙 + 18% : S&amp;P500 :world_map_selector: + 12% : US 10-year Treasury :keycap_10: - 7% : Dollar Index :heavy_dollar_sign: - 22% : Crude Oil :oil_drum_selector:


In [ ]:
def normalize_unicode(text):
    if not isinstance(text, str):
        return ""

    # Zero Width Joiner entfernen
    text = text.replace("\u200d", "")

    # Variation Selectors entfernen
    text = rex.sub(r"[\uFE0E\uFE0F]", "", text)

    return text

In [ ]:
df["text"] = df["text"].apply(normalize_unicode)

In [ ]:
def compress_single_token_runs(text, k_keep=2, min_repeats=3):
    """
    :fire::fire::fire::fire:
    -> :fire::fire:{x4}
    (bei k_keep=2)
    """
    if not isinstance(text, str):
        return ""

    pattern = re.compile(
        r'(:[a-z0-9_]+:)(\1{' + str(min_repeats-1) + r',})'
    )

    def repl(match):
        tok = match.group(1)
        full = match.group(0)
        count = full.count(tok)

        # Erste k behalten
        kept = tok * min(k_keep, count)
        return f"{kept}{{x{count}}}"

    return pattern.sub(repl, text)

In [ ]:
def compress_emoji_sequence_runs(text, k_keep=1, min_seq_len=2, min_repeats=2):
    """
    Komprimiert wiederholte Emoji-Sequenzen.

    Beispiel:
    A B C A B C A B C
    -> A B C {x3}
    (wenn k_keep=1)
    """
    if not isinstance(text, str):
        return ""

    # Finde alle Emoji-Tokens
    emoji_tokens = re.findall(r':[a-z0-9_]+:', text)
    if not emoji_tokens:
        return text

    # Tokenize in Emoji + Nicht-Emoji
    parts = re.split(r'(:[a-z0-9_]+:)', text)
    tokens = [p for p in parts if p]

    i = 0
    out = []

    while i < len(tokens):
        # Versuche Sequenzen ab Länge min_seq_len
        compressed = False

        for seq_len in range(min_seq_len, 8):  # max pattern length = 7 (heuristic)
            seq = tokens[i:i+seq_len]
            if len(seq) < seq_len:
                continue

            # Zähle Wiederholungen
            count = 1
            j = i + seq_len
            while tokens[j:j+seq_len] == seq:
                count += 1
                j += seq_len

            if count >= min_repeats:
                # k Sequenzen behalten
                kept = seq * min(k_keep, count)
                out.extend(kept)
                out.append(f"{{x{count}}}")
                i = j
                compressed = True
                break

        if not compressed:
            out.append(tokens[i])
            i += 1

    return "".join(out)

In [ ]:
def compress_text_pattern_runs(text, pattern_str, k_keep=1, min_repeats=5, flags=re.IGNORECASE):
    if not isinstance(text, str):
        return ""

    pattern = re.compile(
        f'({pattern_str}){{{min_repeats},}}',
        flags
    )

    def repl(match):
        full = match.group(0)
        token = pattern_str
        count = len(re.findall(pattern_str, full, flags))

        kept = token * min(k_keep, count)

        return f"{kept}{{x{count}}}"

    return pattern.sub(repl, text)


In [ ]:
def strip_emoji_modifiers(text):
    # Entfernt skin tone + gender + variation selector
    text = re.sub(r'_(light|medium-light|medium|medium-dark|dark)_skin_tone', '', text)
    text = re.sub(r'::skin-tone-\d+:', ':', text)
    return text

In [ ]:
# Findet das LETZTE :emoji: im Text
LAST_EMOJI_RE = re.compile(r"(.*)(:[^:\s]+:)(.*)", re.DOTALL)

def remove_last_emoji(text):
    m = LAST_EMOJI_RE.match(text)
    if not m:
        return text, False

    before, emoji, after = m.groups()
    # Entferne genau dieses Emoji
    new_text = before + after
    return new_text, True


def trim_by_removing_last_emoji(text, max_tokens=500, max_iters=1000):
    cur = text

    for _ in range(max_iters):
        tok_len = len(tokenizer.tokenize(cur))
        if tok_len <= max_tokens:
            return cur

        cur, removed = remove_last_emoji(cur)
        if not removed:
            # Kein Emoji mehr → abbrechen
            break

    # Fallback: harter Schnitt auf Token-Level
    toks = tokenizer.tokenize(cur)
    toks = toks[:max_tokens]
    return tokenizer.convert_tokens_to_string(toks)

In [ ]:
def smart_shrink_patterns(text, counter=0):
    if not isinstance(text, str):
        return ""

    if counter>2:
        return text

    tokens = tokenizer.tokenize(text)
    if len(tokens) <= 500:
        return text
    print(len(tokens),text)
    # 1. Einzel-Emoji-Runs
    text = compress_single_token_runs(text, k_keep=2, min_repeats=2)

    # 2. Emoji-Sequenzen
    text = compress_emoji_sequence_runs(text, k_keep=1, min_seq_len=2, min_repeats=2)

    # 3. Hashtag-Spam
    text = compress_text_pattern_runs(text, r'&gt;', k_keep=2, min_repeats=3)

    tokens = tokenizer.tokenize(text)
    if len(tokens) >= 500:
        text= strip_emoji_modifiers(text)
        tokens = tokenizer.tokenize(text)
        if len(tokens) >=512:
              text= trim_by_removing_last_emoji(text)

    return text


In [ ]:
tokenizer = model[0].tokenizer  # BERTweet Tokenizer

In [ ]:
df["text"]= df["text"].apply(smart_shrink_patterns)

618 Bitcoin india :India: me banned hai aur Bitcoin news banned hai aur video banned hai :face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy

In [ ]:
counter=0
for i, text in enumerate(df["text"]):
    tokens = tokenizer.tokenize(text)
    if len(tokens) > 512:
        counter+=1
        print(f"The {i}th Text has {len(tokens)} tokens: {text}")

print(counter)

KeyboardInterrupt: 

In [ ]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'datetime', 'username', 'text'], dtype='object')

In [ ]:
len(df)

15630308

In [ ]:
df.to_csv("bitcoin_tweets_clean_bert.csv")